In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

In [28]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

# file path
file_path = "/content/drive/MyDrive/Online Retail.xlsx"

# Load Excel file
df = pd.read_excel(file_path)

print("✅ Dataset loaded successfully")
df.head()

# Step 1
First data inspection

In [ ]:
# Preview the first 5 rows
df.head()
# Data types and non-null counts
df.info()
# Missing values summary
df.isnull().sum()
# Basic statistical summary (numeric columns)
df.describe()

In [ ]:
# % of missing CustomerID
df['CustomerID'].isnull().mean() * 100


## 🔍 Initial Observations

- CustomerID has a large number of missing values (~25%)
- Quantity contains negative values (possible returns or cancellations)
- UnitPrice has extreme outliers (very high max values)
- Description has missing entries
- Data includes potential cancellations (InvoiceNo starting with 'C')

# STEP 2 — DATA CLEANING (ONLINE RETAIL DATASET)

In [ ]:
# Always keep original safe
df_clean = df.copy()

print("Original shape:", df_clean.shape)

In [ ]:
# Flag cancelled invoices (starts with 'C')
df_clean["Cancelled"] = df_clean["InvoiceNo"].astype(str).str.startswith("C")

# Check distribution
df_clean["Cancelled"].value_counts()

In [ ]:
df_clean = df_clean[~df_clean["Cancelled"]]

print("After removing cancellations:", df_clean.shape)

In [ ]:
df_clean = df_clean[df_clean["Quantity"] > 0]

print("After removing invalid Quantity:", df_clean.shape)

In [ ]:
df_clean = df_clean[df_clean["UnitPrice"] > 0]

print("After removing invalid UnitPrice:", df_clean.shape)

## Handling Missing CustomerID

We do NOT immediately drop missing CustomerID because:

1. It may represent guest checkout users
2. Dropping immediately can bias customer segmentation
3. We may still analyze transaction-level patterns

We consider two options:
- OPTION 1: Drop later for customer-based analysis
- OPTION 2: Flag missing IDs for behavioral comparison

In [ ]:
df_clean["CustomerID_missing"] = df_clean["CustomerID"].isna()

df_clean["CustomerID_missing"].value_counts()

In [ ]:
df_customers = df_clean.dropna(subset=["CustomerID"])

In [ ]:
def data_quality_check(data):
    print("🔍 Data Quality Report")
    print("-"*30)
    print("Shape:", data.shape)
    print("Negative Quantity:", (data["Quantity"] <= 0).sum())
    print("Invalid Price:", (data["UnitPrice"] <= 0).sum())
    print("Cancelled Invoices:", data["InvoiceNo"].astype(str).str.startswith("C").sum())
    print("Missing CustomerID:", data["CustomerID"].isna().sum())

data_quality_check(df_clean)

In [ ]:
# Ensure no negative or zero values remain
print("Quantity min:", df_clean["Quantity"].min())
print("UnitPrice min:", df_clean["UnitPrice"].min())

# Ensure no cancellations remain (if removed)
print("Cancelled remaining:", df_clean["InvoiceNo"].astype(str).str.startswith("C").sum())

In [ ]:
print("FINAL CLEAN DATA SHAPE:", df_clean.shape)

## Why Cleaning Matters

- Removing cancellations ensures revenue analysis is not distorted
- Filtering Quantity <= 0 removes returns and data errors
- Removing invalid prices avoids false revenue inflation/deflation
- Handling CustomerID carefully prevents biased customer segmentation
- Flagging missing IDs preserves potential behavioral insights

## Advanced Insight

Instead of blindly removing data, we separate:
- Transaction-level behavior (all rows)
- Customer-level behavior (clean CustomerID dataset)

This allows multi-dimensional analysis without losing business context.  

## Final Data Quality Summary

After cleaning, the dataset contains 530,104 valid transactions with no cancellations, no negative quantities, and no invalid pricing.

However, 132,220 transactions still have missing CustomerID values. These are retained for transaction-level analysis as they may represent guest or unregistered purchases, which are important for understanding overall sales behavior.

# STEP 3 — FEATURE ENGINEERING (ONLINE RETAIL DATASET)

In [ ]:
# Create TotalSales feature (core business metric)
df_clean["TotalSales"] = df_clean["Quantity"] * df_clean["UnitPrice"]

df_clean[["Quantity", "UnitPrice", "TotalSales"]].head()

In [ ]:
# Convert to datetime (safe step)
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])

# Extract time-based features
df_clean["Year"] = df_clean["InvoiceDate"].dt.year
df_clean["Month"] = df_clean["InvoiceDate"].dt.month
df_clean["DayOfWeek"] = df_clean["InvoiceDate"].dt.dayofweek
df_clean["Hour"] = df_clean["InvoiceDate"].dt.hour
df_clean[["InvoiceDate", "Year", "Month", "DayOfWeek", "Hour"]].head()

In [ ]:
# Revenue by country (important for crosstab analysis)
country_sales = df_clean.groupby("Country")["TotalSales"].sum().sort_values(ascending=False)

country_sales.head(10)

In [ ]:
import matplotlib.pyplot as plt

country_sales.head(10).plot(kind="bar")
plt.title("Top 10 Countries by Sales")
plt.ylabel("Total Sales")
plt.show()

In [ ]:
# 50th percentile threshold
threshold = df_clean["TotalSales"].quantile(0.5)

# Create flag
df_clean["HighValueTransaction"] = df_clean["TotalSales"] > threshold

df_clean["HighValueTransaction"].value_counts()

In [ ]:
# Identify stock codes that appear with negative quantity in original data
returned_items = df[df["Quantity"] < 0]["StockCode"].unique()

# Flag returned products in cleaned data
df_clean["Returned"] = df_clean["StockCode"].isin(returned_items)

df_clean["Returned"].value_counts()

## Why These Features Matter

- TotalSales → core business metric for revenue analysis
- Month / DayOfWeek → enables time-based trend analysis
- Country → allows geographic segmentation and crosstab analysis
- HighValueTransaction → helps identify VIP purchases and percentile analysis
- Returned → allows understanding product return behavior and risk

In [ ]:
df_clean.head()

# STEP 4 — ADVANCED ANALYSIS (ONLINE RETAIL)

## Percentile Analysis (Top 10%)

This analysis identifies the top 10% of customers, products, and countries based on TotalSales.  
It helps detect revenue concentration and key business drivers.

In [ ]:
# TOP 10% CUSTOMERS
customer_sales = df_clean.groupby("CustomerID")["TotalSales"].sum()

top_customers = customer_sales[customer_sales >= customer_sales.quantile(0.90)]

print("Top 10% Customers:", len(top_customers))
top_customers.head()

### Insight
This reveals revenue concentration among a small group of customers, indicating a potential VIP segment that drives a large portion of total revenue.

In [ ]:
top_products = df_clean.groupby("StockCode")["TotalSales"].sum()
top_products = top_products[top_products >= top_products.quantile(0.90)]

print("Top Products:", len(top_products))
top_products.head()

In [ ]:
country_sales = df_clean.groupby("Country")["TotalSales"].sum()
top_countries = country_sales[country_sales >= country_sales.quantile(0.90)]

top_countries

## Cross-Tabulation (Country vs Month)

This shows how average sales vary across countries and months.  
It helps identify seasonal and geographic patterns in customer spending behavior.

In [ ]:
crosstab = pd.crosstab(df_clean["Country"], df_clean["Month"], values=df_clean["TotalSales"], aggfunc="mean")

crosstab.head()

### Insight
This helps detect whether certain countries show seasonal spikes in purchasing behavior (e.g., holiday effects).

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
sns.heatmap(crosstab.fillna(0), cmap="Blues")
plt.title("Avg Sales by Country and Month")
plt.show()

## Cohort Analysis

Customers are grouped based on their first purchase month.  
This helps measure retention over time and customer loyalty patterns.

In [ ]:
df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])

df_clean["CohortMonth"] = df_clean.groupby("CustomerID")["InvoiceDate"].transform("min").dt.to_period("M")
df_clean["OrderMonth"] = df_clean["InvoiceDate"].dt.to_period("M")

cohort_data = df_clean.groupby(["CohortMonth", "OrderMonth"])["CustomerID"].nunique().unstack()

cohort_data.head()

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(cohort_data, cmap="YlGnBu")
plt.title("Customer Retention Cohort Analysis")
plt.show()

### Insight
Retention decreases over time, showing that most customers do not return after their first few months.

## Average Order Value (AOV)

This metric shows how much each customer spends per order on average.  
It helps identify high-value customers.

In [ ]:
customer_aov = df_clean.groupby("CustomerID").agg({
    "TotalSales": "sum",
    "InvoiceNo": "nunique"
})

customer_aov["AOV"] = customer_aov["TotalSales"] / customer_aov["InvoiceNo"]

customer_aov.head()

### Insight
A small group of customers has significantly higher AOV, indicating premium or high-spending behavior.

In [ ]:
customer_aov["Segment"] = pd.qcut(customer_aov["AOV"], q=3, labels=["Low", "Medium", "High"])

customer_aov["Segment"].value_counts()

## Time-Based Patterns

This analysis shows how sales change by day of week and month.  
It helps identify peak shopping times.

In [ ]:
day_sales = df_clean.groupby("DayOfWeek")["TotalSales"].mean()

day_sales

In [ ]:
day_sales.plot(kind="bar")
plt.title("Average Sales by Day of Week")
plt.show()

In [ ]:

# Average sales by hour
hour_sales = df_clean.groupby("Hour")["TotalSales"].mean()

print(hour_sales)

# Plot
import matplotlib.pyplot as plt

hour_sales.plot(kind="bar")
plt.title("Average Sales by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Average TotalSales")
plt.show()

### Time-Based Analysis (Hourly Patterns)

I extracted the hour from transaction timestamps to analyze intra-day purchasing behavior.

The analysis shows that sales peak during mid-day hours, indicating that customers are most active during business hours.

This insight can help businesses optimize promotion timing and operational planning.

### Insight
Sales vary significantly across weekdays and months, suggesting time-based shopping behavior patterns.

In [ ]:
month_sales = df_clean.groupby("Month")["TotalSales"].mean()

month_sales.plot(kind="bar")
plt.title("Average Sales by Month")
plt.show()

## Missing Data Pattern Analysis

This compares behavior between customers with missing IDs and those with valid IDs.

In [ ]:
missing = df.copy()

missing["CustomerID_missing"] = missing["CustomerID"].isna()

missing.groupby("CustomerID_missing")[["Quantity", "UnitPrice"]].mean()

### Insight
Customers with missing IDs tend to have different purchasing behavior, suggesting they may represent guest shoppers.

In [ ]:
missing.groupby("CustomerID_missing")["Country"].value_counts().head(10)

## Advanced Analysis Summary

- Percentiles show strong revenue concentration among top 10% customers and products
- Cross-tabulation reveals seasonal and geographic differences in spending
- Cohort analysis shows declining retention over time
- AOV segmentation identifies high-value customers
- Time-based patterns show clear weekly and monthly sales cycles
- Missing data analysis reveals behavioral differences between guest and registered users